# 25 — Modules and the Training Loop

**Master syllabus coverage:** V2 3.6–3.7

## Why this matters

Correct experiments require more than a forward formula. Modules, batching, mode changes, gradient lifecycle, evaluation, and checkpoints form the reusable skeleton of every later training run.

## Learning objectives

- Define a parameter-owning PyTorch module.
- Build reproducible minibatches with variable final batch size.
- Implement a complete gradient-based training step.
- Evaluate without training side effects and serialize resume state.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. A reusable module owns parameters and computation

`nn.Module` registers submodules and parameters, controls train/eval behavior, moves
state between devices, and produces a `state_dict`. The forward method defines
computation; the optimizer owns update state. Keep data iteration, model definition,
loss, optimization, and evaluation as separate responsibilities.


In [ ]:
import io
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

random.seed(42); np.random.seed(42); torch.manual_seed(42)

class Classifier(torch.nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int) -> None:
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_dim, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

model = Classifier(2, 16)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))


## 2. Minibatches estimate the dataset gradient

Shuffling changes grouping each epoch. A dedicated generator makes order reproducible.
The final batch may be smaller unless `drop_last=True`; model code should not assume a
fixed batch dimension. For language models, batches also group token sequences.


In [ ]:
generator = torch.Generator().manual_seed(42)
points = torch.randn(400, 2, generator=generator)
labels = ((points[:, 0] ** 2 + points[:, 1] ** 2) > 1.0).long()
train_x, val_x = points[:320], points[320:]
train_y, val_y = labels[:320], labels[320:]
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=32,
                    shuffle=True, generator=torch.Generator().manual_seed(7))
first_x, first_y = next(iter(loader))
assert first_x.shape == (32, 2) and first_y.shape == (32,)


## 3. The canonical training step

1. switch to training mode;
2. compute predictions and scalar loss;
3. clear old gradients;
4. backpropagate;
5. optionally inspect or clip gradients;
6. update parameters.

Clearing before or after the step can both work if done consistently. `set_to_none=True`
avoids unnecessary zero fills and makes missing gradients easier to notice.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.02, weight_decay=1e-3)
history = []
for epoch in range(25):
    model.train()
    total_loss, total_items = 0.0, 0
    for batch_x, batch_y in loader:
        logits = model(batch_x)
        loss = F.cross_entropy(logits, batch_y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        assert torch.isfinite(gradient_norm)
        optimizer.step()
        total_loss += loss.item() * batch_x.shape[0]
        total_items += batch_x.shape[0]
    history.append(total_loss / total_items)

print("training loss:", history[0], "→", history[-1])
assert history[-1] < history[0]


## 4. Evaluation is a distinct, side-effect-free phase

`model.eval()` disables dropout randomness and changes batch-normalization behavior.
`torch.inference_mode()` avoids gradient recording and additional autograd overhead.
Weighting batch metrics by batch size prevents a small final batch from counting equally.


In [ ]:
def evaluate(module, features, targets):
    module.eval()
    with torch.inference_mode():
        logits = module(features)
        loss = F.cross_entropy(logits, targets)
        accuracy = (logits.argmax(dim=-1) == targets).float().mean()
    return loss.item(), accuracy.item()

val_loss, val_accuracy = evaluate(model, val_x, val_y)
print(f"validation loss={val_loss:.3f}, accuracy={val_accuracy:.3f}")
assert val_accuracy > 0.85


## 5. A checkpoint is more than weights

Exact resume needs model state, optimizer state, current step/epoch, configuration,
scheduler/scaler state when used, and random-number-generator states. Production
experiments also record Git commit, dataset identity, device, metrics, and launch command.


In [ ]:
checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "epoch": 24,
    "config": {"input_dim": 2, "hidden_dim": 16, "seed": 42},
    "torch_rng_state": torch.get_rng_state(),
}
buffer = io.BytesIO()
torch.save(checkpoint, buffer)
buffer.seek(0)
restored = torch.load(buffer, weights_only=False)
assert restored["epoch"] == 24 and restored["config"]["seed"] == 42
print("in-memory checkpoint bytes:", buffer.getbuffer().nbytes)


## Failure modes to recognize

- **Gradients never cleared:** every batch changes the effective update.
- **Eval mode omitted:** dropout makes metrics stochastic and batch norm mutates/uses the wrong state.
- **Loss averaged by batches:** a small last batch receives too much weight.
- **Weights-only checkpoint:** optimizer trajectory and exact resume are lost.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Refactor the loop into `train_epoch` and test that every parameter receives a finite gradient.
2. Resume from the in-memory checkpoint into a new model and verify identical evaluation logits.
3. Record a minimal experiment manifest with commit, seed, data, config, metric, device, and command.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can train, evaluate, checkpoint, and exactly describe the mutable state involved in an experiment.

**Next:** 26 — Initialization and gradient flow.
